# Proposed validation — review before it runs

**What this measures:** The requester's question is about the mechanism the PR adds and `main` cannot run at all — the pure-python `TokenVocabAligner` — so this is a single-arm capability test, not a throughput run (the repo's benchmark harness measures serving throughput with model weights, which cannot observe alignment fidelity and which the user guidance deliberately descopes). The target metric `roundtrip_text_survival_rate` is the fraction of target-vocabulary ids whose decoded string is identical after the target→draft→target round trip through the PR's own `from_tokenizers` tables — by the exact-match-first bijection this equals the exact-string coverage |T∩D|/|T|, with a provable byte-token floor (256/151665) and a pigeonhole ceiling (49152/151665), while the guardrail demands covered tokens round-trip with identity exactly 1.0.

**Target metric:** `roundtrip_text_survival_rate`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_cross_vocab_alignment.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [1]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "d7a4ddd326144492a3fa78f814a96cc63866fa1d"
seed = 0


## Execution context

The cells below are the script at `eval/eval_cross_vocab_alignment.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [3]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_cross_vocab_alignment.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

[remyx] cwd /workspace/target_repo | script /workspace/target_repo/eval/eval_cross_vocab_alignment.py


In [4]:
# Copyright (c) OpenMMLab. All rights reserved.
"""Aligner-fidelity benchmark for the cross-vocabulary speculative proposer.

User-guidance scope: with Qwen/Qwen2.5-7B-Instruct as target and
HuggingFaceTB/SmolLM2-135M-Instruct as draft (tokenizer files only, no model
weights), does text survive the target -> draft -> target round trip through
the PR's TokenVocabAligner intact, and what share of each vocabulary has no
counterpart in the other?

The aligner module is loaded BY PATH: it is pure python by PR design, so no
torch / lmdeploy package import is pulled in (the eval env guarantees only
transformers + tokenizers). The tables are a deterministic function of the two
tokenizers; no sampling, no RNG, no wall-clock metrics.

Two arms. On the PR head the diff-added aligner module exists and the real
measurement runs. On the baseline arm the module is absent: the defensive
import below fails cleanly (ImportError/AttributeError/FileNotFoundError only)
and zeroed metrics with identical keys are emitted, so the script runs without
crashing on both refs. Question kind is capability (baseline: none): a
tokenizer download failure or a present-but-broken aligner file still raises
loudly with a traceback rather than emitting stand-in numbers — no blanket
try/except wraps the measurement.
"""

"Aligner-fidelity benchmark for the cross-vocabulary speculative proposer.\n\nUser-guidance scope: with Qwen/Qwen2.5-7B-Instruct as target and\nHuggingFaceTB/SmolLM2-135M-Instruct as draft (tokenizer files only, no model\nweights), does text survive the target -> draft -> target round trip through\nthe PR's TokenVocabAligner intact, and what share of each vocabulary has no\ncounterpart in the other?\n\nThe aligner module is loaded BY PATH: it is pure python by PR design, so no\ntorch / lmdeploy package import is pulled in (the eval env guarantees only\ntransformers + tokenizers). The tables are a deterministic function of the two\ntokenizers; no sampling, no RNG, no wall-clock metrics.\n\nTwo arms. On the PR head the diff-added aligner module exists and the real\nmeasurement runs. On the baseline arm the module is absent: the defensive\nimport below fails cleanly (ImportError/AttributeError/FileNotFoundError only)\nand zeroed metrics with identical keys are emitted, so the script runs 

In [5]:
import argparse
import importlib.util
import json
import os
import sys
from functools import lru_cache

In [6]:
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, REPO_ROOT)
ALIGNER_PATH = os.path.join(REPO_ROOT, 'lmdeploy', 'pytorch', 'spec_decode', 'vocab_alignment.py')
TARGET_MODEL = 'Qwen/Qwen2.5-7B-Instruct'             # ~151,665-token vocab
DRAFT_MODEL = 'HuggingFaceTB/SmolLM2-135M-Instruct'   # 49,152-token vocab

# Fixed probe corpus: ASCII prose, code, JSON, ChatML markup, CJK, multilingual, emoji.
PROBES = [
    'Hello, world!',
    'The quick brown fox jumps over the lazy dog.',
    'def add(a, b):\n    return a + b\n',
    '{"name": "lmdeploy", "tags": ["spec", "decode"], "n": 3}',
    '<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\n',
    '東京は日本の首都です。北京も大都市だ。',
    'Bonjour, ça va? ¿Cómo estás? — naïve façade, über alles.',
    'Emoji test: 🚀🔥✨ — done.',
]

In [7]:
def load_aligner_class():
    """Load the PR's vocab_alignment.py by file path (the symbol this diff
    adds). Returns None when the module/symbol is absent (baseline arm:
    FileNotFoundError is the by-path analog of ImportError). A present but
    broken file (SyntaxError, etc.) raises here so the failure is visible."""
    try:
        spec = importlib.util.spec_from_file_location('vocab_alignment', ALIGNER_PATH)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module.TokenVocabAligner
    except (ImportError, AttributeError, FileNotFoundError):
        return None

In [8]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--variant', default=None)  # accepted, ignored
    parser.add_argument('--ref', default=None)      # accepted, ignored
    parser.add_argument('--seed', default=None)     # accepted, ignored
    parser.parse_known_args()
    smoke = os.environ.get('REMYX_SMOKE') == '1'

    aligner_cls = load_aligner_class()
    if aligner_cls is None:
        # Baseline arm: the cross-vocab aligner does not exist pre-change, so
        # the capability is absent -> zeroed/degraded metrics, same JSON keys.
        print(json.dumps({
            'roundtrip_text_survival_rate': 0.0,
            'covered_roundtrip_identity': 0.0,
            'target_vocab_no_counterpart_share': 0.0,
            'draft_vocab_no_counterpart_share': 0.0,
            'intersection_count_mismatch': 0,
            'probe_text_survival_rate': 0.0,
        }))
        return

    from transformers import AutoTokenizer
    target_tok = AutoTokenizer.from_pretrained(TARGET_MODEL, trust_remote_code=True)
    draft_tok = AutoTokenizer.from_pretrained(DRAFT_MODEL, trust_remote_code=True)

    # Exercise the real PR code exactly as CrossVocabProposer builds it.
    aligner = aligner_cls.from_tokenizers(draft_tok, target_tok)
    t2d, d2t = aligner.target_to_draft, aligner.draft_to_target
    n_t, n_d = len(t2d), len(d2t)

    # Counterpart coverage is judged on vocab-table strings
    # (convert_ids_to_tokens), NOT lossy decode: these strings are unique per
    # id, so the exact-match stage is a bijection on the shared string set and
    # the pigeonhole bound applies (covered target ids <= 49,152). Round-trip
    # survival is judged on decoded strings so that partial-UTF-8 byte-token
    # decode collapse (U+FFFD) cannot swap ids while altering text.
    conv_t = target_tok.convert_ids_to_tokens(list(range(n_t)))
    conv_d = draft_tok.convert_ids_to_tokens(list(range(n_d)))

    @lru_cache(maxsize=None)
    def dec_t(i):
        return target_tok.decode([i], clean_up_tokenization_spaces=False)

    lim_t = min(n_t, 8192) if smoke else n_t  # smoke: identical build/code path, smaller sweep
    lim_d = min(n_d, 4096) if smoke else n_d

    survive = covered_t = identity_ok = 0
    for t in range(lim_t):
        d = t2d[t]
        t_back = d2t[d]
        covered = conv_d[d] == conv_t[t]
        intact = dec_t(t_back) == dec_t(t)
        survive += int(intact)
        covered_t += int(covered)
        identity_ok += int(covered and intact)

    covered_d = sum(1 for d in range(lim_d) if conv_t[d2t[d]] == conv_d[d])

    probes = PROBES[:2] if smoke else PROBES
    probe_ok = 0
    for probe in probes:
        ids = target_tok.encode(probe, add_special_tokens=False)
        back = [d2t[t2d[t]] for t in ids]
        probe_ok += int(target_tok.decode(back, clean_up_tokenization_spaces=False) == probe)

    metrics = {
        # Floor: >= 256 shared single-byte tokens / 151,665 = 0.00169.
        # Pigeonhole ceiling: 49,152 / 151,665 = 0.3241; expected ~0.1-0.3.
        'roundtrip_text_survival_rate': round(survive / lim_t, 6),
        # Exact-match-first bijection on shared strings: must be exactly 1.0.
        'covered_roundtrip_identity': round(identity_ok / covered_t, 6) if covered_t else 0.0,
        # Pigeonhole floor: >= 1 - 49,152/151,665 = 0.6759 regardless of content.
        'target_vocab_no_counterpart_share': round(1.0 - covered_t / lim_t, 6),
        # Ceiling: <= 1 - 256/49,152 = 0.99479 (both vocabs hold all 256 byte tokens).
        'draft_vocab_no_counterpart_share': round(1.0 - covered_d / lim_d, 6),
        # Two-sided consistency: both covered counts equal |T ∩ D|, mismatch is 0.
        'intersection_count_mismatch': abs(covered_t - covered_d),
        # Fixed probes; at minimum the short ASCII ones ride the shared subword core.
        'probe_text_survival_rate': round(probe_ok / len(probes), 6),
    }
    print(json.dumps(metrics))

if __name__ == '__main__':
    main()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


{"roundtrip_text_survival_rate": 0.267168, "covered_roundtrip_identity": 1.0, "target_vocab_no_counterpart_share": 0.743303, "draft_vocab_no_counterpart_share": 0.207926, "intersection_count_mismatch": 0, "probe_text_survival_rate": 0.375}


## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "With Qwen/Qwen2.5-7B-Instruct as target and HuggingFaceTB/SmolLM2-135M-Instruct as draft, does text survive the round trip through the aligner intact, and what share of each vocabulary has no counterpart in the other?"
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: cross-vocab-aligner-fidelity
    suite: "eval/eval_cross_vocab_alignment.py"
    packages: ["transformers", "tokenizers"]
    scorer: roundtrip_text_survival_rate
    baseline: none
    metrics:
      - name: roundtrip_text_survival_rate
        direction: max
        # Floor: both are byte-level BPEs sharing all 256 single-byte tokens, so
        # survival >= 256/151665 = 0.00169; threshold sits just under that floor.
        # Pigeonhole ceiling: <= 49152/151665 = 0.3241. Expected measured ~0.1-0.3.
        threshold: 0.0015
        role: target
      - name: covered_roundtrip_identity
        direction: max
        # Exact-match-first is a bijection on the shared string set: every target
        # id whose mapped draft id has the SAME string MUST round-trip back to the
        # same string. Identity is exact — anything < 1.0 is an aligner defect.
        threshold: 1.0
        role: guardrail
      - name: target_vocab_no_counterpart_share
        direction: max
        # Pigeonhole floor: >= 1 - 49152/151665 = 0.6759 regardless of content
        # (at least 102,513 Qwen tokens necessarily have no exact counterpart).
        threshold: 0.6759
        role: diagnostic
      - name: draft_vocab_no_counterpart_share
        direction: min
        # Ceiling: |T∩D| >= 256 shared byte tokens, so the uncovered share of the
        # 49,152-token draft vocab is <= 1 - 256/49152 = 0.99479.
        threshold: 0.9948
        role: diagnostic
      - name: intersection_count_mismatch
        direction: min
        # Consistency: covered_target_count must equal covered_draft_count (both
        # equal |T∩D|); the mismatch is exactly 0 for a correct two-sided build.
        threshold: 0
        role: diagnostic
      - name: probe_text_survival_rate
        direction: max
        # Fixed probes (ASCII prose, code/JSON, ChatML, CJK, emoji): at minimum
        # the plain-ASCII probes ride the shared ASCII-subword core, so a working
        # exact-match stage keeps some probes byte-identical. No provable floor
        # beyond that; this is reported, not gated.
        threshold: 0.1
        role: diagnostic
    policy: {guardrail_veto: true}
    held_constant:
      - "Same tokenizer pair on every run: Qwen/Qwen2.5-7B-Instruct as target, HuggingFaceTB/SmolLM2-135M-Instruct as draft; tokenizer files only, no model weights"
      - "Fixed probe corpus inline in the script: ASCII prose, code/JSON, ChatML markup, CJK/multilingual, emoji"
      - "The aligner tables are a deterministic pure-python function of the two tokenizers (TokenVocabAligner.from_tokenizers); no sampling, no RNG"
      - "Round trip judged on decoded strings (clean_up_tokenization_spaces=False on both sides) so partial-UTF-8 byte-token decode collapse cannot swap ids while preserving text"
    avoid:
      - "No wall-clock metrics: shared CPU makes timing noise and the claim is fidelity, not speed"
      - "No model-weight downloads, no GPU, no serving run: the paper's 1.57x speedup is out of scope because the PR replaces per-sequence DTW with static vocab tables"
      - "Do not reimplement the DTW/exact-match logic inline: the script loads the PR's lmdeploy/pytorch/spec_decode/vocab_alignment.py by path (pure python by PR design, so no torch import is pulled in)"
      - "HF tokenizer revisions resolve to the default branch: tension with the pinned-revision rule recorded here; both repos' tokenizer artifacts are frozen since release and only tokenizer files are loaded, so the measurement is reproducible in practice"
      - "If from_tokenizers prunes DTW candidates poorly the one-time table build dominates runtime; smoke mode runs the identical build (same code path and data source), so a pathological build cost surfaces as a timeout and is itself a finding"
    compute: {tier: cpu}
    provenance:
      roundtrip_text_survival_rate: "user_guidance"
      covered_roundtrip_identity: "user_guidance + arXiv:2510.15545 mechanism (exact-match-first bijection on the shared string set)"
      target_vocab_no_counterpart_share: "user_guidance"
      draft_vocab_no_counterpart_share: "user_guidance"
      intersection_count_mismatch: "claim_analysis arithmetic (both covered counts equal |T∩D|)"
      probe_text_survival_rate: "user_guidance (does text survive the round trip intact)"
      held_constant: "claim_analysis protocol (offline, CPU-only, no weights, pinned tokenizers, fixed probes)"
      suite: "synthesized"
      compute: "claim_analysis protocol (aligner is pure python by PR design; CPU suffices)"
```